# Loan Delinquency Risk Analysis

### Background

Meridian Trust Bank has seen its Non-Performing Assets (NPAs) climb steadily, and a significant share
of that stems from loans issued to retail (individual) borrowers. To get ahead of this, the bank's
Chief Risk Officer wants a data-driven framework for deciding which individual applicants should be
approved for a loan, with the goal of cutting down on future delinquencies. As a senior data scientist
on the risk team, you have been asked to lead this analysis.

### Goal
Determine which applicant characteristics best predict loan delinquency so the bank can set smarter
approval criteria.

### Key question
Which factors are most associated with a borrower becoming delinquent on their loan?

### Data Dictionary
* ID: Unique customer identifier
* isDelinquent: Target flag - 1 if the customer is delinquent, 0 otherwise
* term: Loan term, in months
* gender: Borrower's gender
* age: Borrower's age bracket
* purpose: Reason the loan was taken out
* home_ownership: Borrower's housing situation
* FICO: Borrower's credit bureau (FICO) score bracket

### Glossary
Transactor - Someone who pays their balance in full and on time every cycle.

Revolver - Someone who only pays the minimum due and carries a balance forward.

Delinquent - Behind on payments; has missed even the minimum due amount.

Defaulter - A delinquent borrower who has remained unpaid long enough that the lender classifies the loan as in default.

Risk Analytics - The practice, common across banking and finance, of quantifying and managing customer/borrower risk.


### Step 1: Import Libraries

In [ ]:
# Libraries for data handling
import numpy as np
import pandas as pd

# Libraries for plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Model tuning, scoring and data-splitting utilities
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score,
)

# Classifier
from sklearn.linear_model import LogisticRegression

# Statistical testing
import scipy.stats as stats

# Class-imbalance handling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Silence warnings for a cleaner notebook
import warnings

warnings.filterwarnings("ignore")


### Step 2: Load the Data

In [ ]:
raw_loans = pd.read_csv("Loan_Delinquent_Dataset.csv")

In [ ]:
# Row and column count
raw_loans.shape

### Step 3: First Look at the Data

In [ ]:
# Work on a copy so the original import stays untouched
loan_df = raw_loans.copy()

In [ ]:
# First 5 records
loan_df.head()

In [ ]:
# Last 5 records
loan_df.tail()

In [ ]:
# Column data types and non-null counts
loan_df.info()

## Step 4: Correcting Column Data Types

`term`, `gender`, `purpose`, `home_ownership`, `age` and `FICO` are stored as generic objects; converting
them to `category` dtype is more memory-efficient and semantically correct since these are all categorical fields.


In [ ]:
loan_df["term"] = loan_df["term"].astype("category")
loan_df["gender"] = loan_df["gender"].astype("category")
loan_df["purpose"] = loan_df["purpose"].astype("category")
loan_df["home_ownership"] = loan_df["home_ownership"].astype("category")
loan_df["age"] = loan_df["age"].astype("category")
loan_df["FICO"] = loan_df["FICO"].astype("category")
loan_df["isDelinquent"] = loan_df["isDelinquent"].astype("category")

In [ ]:
loan_df.info()

`Memory usage drops noticeably once the object columns are converted to category dtype.`

**Observations**
* Apart from `ID`, every remaining column is categorical.


In [ ]:
# Check for duplicate rows
loan_df.duplicated().sum()

In [ ]:
# Check for missing values
loan_df.isnull().sum()

- The dataset has no missing values.

In [ ]:
loan_df.describe(include="all")

**Observations**

* The majority class for `isDelinquent` is 1 (delinquent).
* Most loans run for a 36-month term.
* Male applicants outnumber female applicants.
* House loans are the most common purpose.
* Mortgage is the most common home-ownership status.
* The 20-25 age bracket applies for loans most often.
* Most customers fall in the 300-500 FICO bracket.


## Step 5: Data Cleaning

In [ ]:
# ID should be unique for every row
loan_df["ID"].nunique()

* Since `ID` is unique per row it carries no predictive signal and can be dropped.

In [ ]:
loan_df.drop(["ID"], axis=1, inplace=True)

In [ ]:
# Inspect distinct values under purpose
loan_df["purpose"].unique()

In [ ]:
# 'other' and 'Other' are the same category, so merge them
loan_df["purpose"].replace("other", "Other", inplace=True)

In [ ]:
loan_df["purpose"].unique()

## Step 6: Exploratory Data Analysis (EDA)

### Univariate Analysis

In [ ]:
# Reusable helper to draw a count/percentage-labeled bar plot for one categorical column
def plot_labeled_bar(df, column, show_pct=False, top_n=None):
    """
    Draws a bar chart for a categorical column with the count (or percentage) labeled on each bar.

    df: source dataframe
    column: name of the categorical column to plot
    show_pct: label bars with percentages instead of raw counts when True
    top_n: only plot the top_n most frequent categories (None plots every category)
    """

    row_count = len(df[column])  # total number of observations
    category_count = df[column].nunique()
    if top_n is None:
        plt.figure(figsize=(category_count + 1, 5))
    else:
        plt.figure(figsize=(top_n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=df,
        x=column,
        palette="Paired",
        order=df[column].value_counts().index[:top_n].sort_values(),
    )

    for bar in ax.patches:
        if show_pct:
            bar_label = "{:.1f}%".format(
                100 * bar.get_height() / row_count
            )  # share of total for this category
        else:
            bar_label = bar.get_height()  # raw count for this category

        x_pos = bar.get_x() + bar.get_width() / 2
        y_pos = bar.get_height()

        ax.annotate(
            bar_label,
            (x_pos, y_pos),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )

    plt.show()

#### isDelinquent

In [ ]:
plot_labeled_bar(loan_df, "isDelinquent")

* Roughly 66% of borrowers in the data are delinquent.

#### term

In [ ]:
plot_labeled_bar(loan_df, "term")

* About 91.7% of loans have a 36-month term.

#### gender

In [ ]:
plot_labeled_bar(loan_df, "gender")

* Male applicants (56.8%) outnumber female applicants (43.2%).

#### purpose

In [ ]:
plot_labeled_bar(loan_df, "purpose")

* House loans dominate the purpose field (59.7%), followed by car loans (18%).
* The `other`/`Other` duplication noted earlier has already been merged.

#### home_ownership

In [ ]:
plot_labeled_bar(loan_df, "home_ownership")

* Fewer than 10% of applicants outright own their home; most either have a mortgage or rent.

#### age

In [ ]:
plot_labeled_bar(loan_df, "age")

* The 20-25 and >25 age brackets are represented in roughly equal proportion.

#### FICO

In [ ]:
plot_labeled_bar(loan_df, "FICO")

* The 300-500 FICO bracket (55.2%) is slightly more common than the >500 bracket (44.8%).

In [ ]:
# Reusable helper to draw a normalized stacked bar chart of predictor vs. target
def plot_stacked_bar(df, predictor, outcome):
    """
    Prints the raw cross-tab counts and draws a normalized stacked bar chart.

    df: source dataframe
    predictor: independent/explanatory column
    outcome: dependent/target column
    """
    category_count = df[predictor].nunique()
    sort_level = df[outcome].value_counts().index[-1]
    raw_counts = pd.crosstab(df[predictor], df[outcome], margins=True).sort_values(
        by=sort_level, ascending=False
    )
    print(raw_counts)
    print("-" * 120)
    normalized = pd.crosstab(df[predictor], df[outcome], normalize="index").sort_values(
        by=sort_level, ascending=False
    )
    normalized.plot(kind="bar", stacked=True, figsize=(category_count + 1, 5))
    plt.legend(
        loc="lower left",
        frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
plot_stacked_bar(loan_df, "term", "isDelinquent")

* Delinquency is most common among borrowers with a 36-month loan term.

In [ ]:
plot_stacked_bar(loan_df, "gender", "isDelinquent")

* Delinquency rates are fairly similar across genders.

In [ ]:
plot_stacked_bar(loan_df, "purpose", "isDelinquent")

* House loans account for the largest share of delinquencies, followed by car and personal loans.

In [ ]:
plot_stacked_bar(loan_df, "home_ownership", "isDelinquent")

* Borrowers who own their home outright are less delinquent than renters or those with a mortgage.

In [ ]:
plot_stacked_bar(loan_df, "age", "isDelinquent")

* The 20-25 age bracket shows higher delinquency.

In [ ]:
plot_stacked_bar(loan_df, "FICO", "isDelinquent")

* A FICO score above 500 is associated with a much lower delinquency rate than the 300-500 bracket.

### Key Takeaways So Far
* FICO bracket and loan term look like the strongest signals for delinquency.
* The remaining predictors appear weaker on their own (a chi-square test can confirm statistical significance).


### Do any other fields correlate with FICO bracket itself?

In [ ]:
plot_stacked_bar(loan_df, "home_ownership", "FICO")

In [ ]:
plot_stacked_bar(loan_df, "age", "FICO")

In [ ]:
plot_stacked_bar(loan_df, "gender", "FICO")

## Observations

1. `home_ownership` and `gender` show a mild association with FICO bracket.
2. `age` shows a much stronger association with FICO bracket.


### Testing Statistical Significance

The chi-square test of independence checks whether two categorical variables are associated.

**Null hypothesis (H0):** The two variables are independent (no association).
**Alternate hypothesis (H1):** The two variables are associated.


In [ ]:
contingency_tbl = pd.crosstab(
    loan_df["FICO"], loan_df["home_ownership"]
)  # cross-tab of FICO bracket vs. home ownership

null_hyp = "FICO bracket has no relationship with home ownership"
alt_hyp = "FICO bracket has a relationship with home ownership"

chi_stat, p_val, deg_free, expected_freq = stats.chi2_contingency(contingency_tbl)

if p_val < 0.05:  # 5% significance level
    print(f"{alt_hyp} as the p_value ({p_val.round(3)}) < 0.05")
else:
    print(f"{null_hyp} as the p_value ({p_val.round(3)}) > 0.05")

In [ ]:
contingency_tbl = pd.crosstab(
    loan_df["FICO"], loan_df["gender"]
)  # cross-tab of FICO bracket vs. gender

null_hyp = "FICO bracket has no relationship with gender"
alt_hyp = "FICO bracket has a relationship with gender"

chi_stat, p_val, deg_free, expected_freq = stats.chi2_contingency(contingency_tbl)

if p_val < 0.05:
    print(f"{alt_hyp} as the p_value ({p_val.round(3)}) < 0.05")
else:
    print(f"{null_hyp} as the p_value ({p_val.round(3)}) > 0.05")

In [ ]:
contingency_tbl = pd.crosstab(
    loan_df["FICO"], loan_df["age"]
)  # cross-tab of FICO bracket vs. age

null_hyp = "FICO bracket has no relationship with age"
alt_hyp = "FICO bracket has a relationship with age"

chi_stat, p_val, deg_free, expected_freq = stats.chi2_contingency(contingency_tbl)

if p_val < 0.05:
    print(f"{alt_hyp} as the p_value ({p_val.round(3)}) < 0.05")
else:
    print(f"{null_hyp} as the p_value ({p_val.round(3)}) > 0.05")

## Observations

* Every test returns a p-value under 0.01, so all three relationships seen in the plots are statistically significant.
* FICO bracket is associated with home ownership - borrowers with a mortgage tend to have higher FICO scores than
  those who own their home outright (a bit counterintuitive).
* FICO bracket is associated with gender - a larger share of female borrowers fall in the >500 bracket than male borrowers.
* FICO bracket is associated with age - borrowers over 25 tend to have higher FICO scores than those aged 20-25.


## Step 7: Preparing Data for Modeling

In [ ]:
predictors = loan_df.drop(["isDelinquent"], axis=1)
target = loan_df["isDelinquent"]

In [ ]:
# One-hot encode the categorical predictors (cast to int so downstream libraries
# that don't play well with bool dtype, e.g. SMOTE, work without issues)
predictors = pd.get_dummies(predictors, drop_first=True).astype(int)
predictors.head()

In [ ]:
# First split off a hold-out test set, then carve a validation set out of the remainder
temp_X, test_X, temp_y, test_y = train_test_split(
    predictors, target, test_size=0.2, random_state=1, stratify=target
)

train_X, val_X, train_y, val_y = train_test_split(
    temp_X, temp_y, test_size=0.25, random_state=1, stratify=temp_y
)
print(train_X.shape, val_X.shape, test_X.shape)

In [ ]:
print("Number of rows in train data =", train_X.shape[0])
print("Number of rows in validation data =", val_X.shape[0])
print("Number of rows in test data =", test_X.shape[0])

## Step 8: Building the Model

### How should we judge model performance?

**What is the bank trying to avoid?**
* Two kinds of loss are possible here:
   * The bank lends to a customer who never pays it back.
   * The bank turns away a customer who would actually have repaid - an opportunity cost.

**Which of these hurts more?**
* Lending to someone who can't repay is the costlier mistake.

**Because the priority is catching delinquent borrowers, Recall is the metric to optimize for, not plain accuracy.**

* Recall = True Positives / Actual Positives. A high recall means few false negatives - i.e., we rarely mislabel
  a delinquent borrower as safe.


In [ ]:
# Computes accuracy, recall, precision and F1 for a fitted sklearn classifier
def get_classification_metrics(model, X_data, y_data):
    """
    Returns a one-row dataframe of accuracy, recall, precision and F1 for the given model/data.

    model: fitted classifier
    X_data: feature matrix to score
    y_data: true labels for X_data
    """

    y_hat = model.predict(X_data)

    acc = accuracy_score(y_data, y_hat)
    rec = recall_score(y_data, y_hat)
    prec = precision_score(y_data, y_hat)
    f1 = f1_score(y_data, y_hat)

    metrics_df = pd.DataFrame(
        {
            "Accuracy": acc,
            "Recall": rec,
            "Precision": prec,
            "F1": f1,
        },
        index=[0],
    )

    return metrics_df

In [ ]:
# Plots a confusion matrix annotated with counts and percentages
def plot_confusion_heatmap(model, X_data, y_data):
    """
    Draws a heatmap confusion matrix for the given model/data, annotated with count and % of total.

    model: fitted classifier
    X_data: feature matrix to score
    y_data: true labels for X_data
    """
    y_hat = model.predict(X_data)
    cm = confusion_matrix(y_data, y_hat)
    cell_labels = np.asarray(
        [
            ["{0:0.0f}".format(val) + "\n{0:.2%}".format(val / cm.flatten().sum())]
            for val in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=cell_labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

### Baseline Logistic Regression

In [ ]:
base_logit = LogisticRegression(random_state=1)
base_logit.fit(train_X, train_y)

**Cross-validated recall check**

- K-fold cross-validation splits the data into k stratified folds and rotates which fold is held out for
  validation, giving a more robust read on how the model generalizes.


In [ ]:
cv_metric = "recall"
cv_splitter = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=1
)
base_cv_scores = cross_val_score(
    estimator=base_logit, X=train_X, y=train_y, scoring=cv_metric, cv=cv_splitter
)
plt.boxplot(base_cv_scores)
plt.show()

* Recall on the training folds ranges roughly between 0.86 and 0.87.
* Next, check performance on the validation set.


In [ ]:
base_train_metrics = get_classification_metrics(base_logit, train_X, train_y)
print("Training performance:")
base_train_metrics

In [ ]:
base_val_metrics = get_classification_metrics(base_logit, val_X, val_y)
print("Validation performance:")
base_val_metrics

In [ ]:
plot_confusion_heatmap(base_logit, val_X, val_y)

* The baseline model generalizes reasonably well between train and validation.
* Next, try oversampling to see if it helps further.


### Oversampling the Training Data with SMOTE

In [ ]:
print("Before UpSampling, counts of label 'Yes': {}".format(sum(train_y == 1)))
print("Before UpSampling, counts of label 'No': {} \n".format(sum(train_y == 0)))

smote_sampler = SMOTE(
    sampling_strategy=1, k_neighbors=5, random_state=1
)
train_X_over, train_y_over = smote_sampler.fit_resample(train_X, train_y)


print("After UpSampling, counts of label 'Yes': {}".format(sum(train_y_over == 1)))
print("After UpSampling, counts of label 'No': {} \n".format(sum(train_y_over == 0)))


print("After UpSampling, the shape of train_X: {}".format(train_X_over.shape))
print("After UpSampling, the shape of train_y: {} \n".format(train_y_over.shape))

### Logistic Regression on Oversampled Data

In [ ]:
oversampled_logit = LogisticRegression(random_state=1)

oversampled_logit.fit(train_X_over, train_y_over)

**Cross-validated recall check**

- Same K-fold approach as before, now run on the oversampled training data.


In [ ]:
cv_metric = "recall"
cv_splitter = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=1
)
over_cv_scores = cross_val_score(
    estimator=oversampled_logit, X=train_X_over, y=train_y_over, scoring=cv_metric, cv=cv_splitter
)
plt.boxplot(over_cv_scores)
plt.show()

* Training-fold recall now ranges roughly between 0.69 and 0.70.
* Next, check validation performance.


In [ ]:
over_train_metrics = get_classification_metrics(
    oversampled_logit, train_X_over, train_y_over
)
print("Training performance:")
over_train_metrics

In [ ]:
over_val_metrics = get_classification_metrics(
    oversampled_logit, val_X, val_y
)
print("validation performance:")
over_val_metrics

In [ ]:
plot_confusion_heatmap(oversampled_logit, val_X, val_y)

* Training performance improved, but the validation set doesn't see the same gain.
* This is a sign of overfitting.
* Two things worth trying next:

  a) Regularization, to rein in overfitting

  b) Undersampling the majority class instead, to rebalance the training data differently


### Regularization

In [ ]:
# Choose the classifier and a solver that supports L1/L2 penalties
tuned_logit_base = LogisticRegression(random_state=1, solver="saga")

# Candidate values for the regularization strength C
param_grid = {"C": np.arange(0.1, 1.1, 0.1)}

# Search for the best C via cross-validated recall
grid_search = GridSearchCV(tuned_logit_base, param_grid, scoring="recall")
grid_search = grid_search.fit(train_X_over, train_y_over)

# Keep the best-performing estimator
tuned_logit = grid_search.best_estimator_

# Refit on the oversampled training data
tuned_logit.fit(train_X_over, train_y_over)

In [ ]:
tuned_train_metrics = get_classification_metrics(
    tuned_logit, train_X_over, train_y_over
)
print("Training performance:")
tuned_train_metrics

In [ ]:
tuned_val_metrics = get_classification_metrics(
    tuned_logit, val_X, val_y
)
print("Validation performance:")
tuned_val_metrics

In [ ]:
plot_confusion_heatmap(tuned_logit, val_X, val_y)

* Recall looks strong here, but it's close to what the oversampled model already achieved.

### Undersampling the Training Data with RandomUnderSampler

In [ ]:
undersampler = RandomUnderSampler(random_state=1)
train_X_under, train_y_under = undersampler.fit_resample(train_X, train_y)

In [ ]:
print("Before Under Sampling, counts of label 'Yes': {}".format(sum(train_y == 1)))
print("Before Under Sampling, counts of label 'No': {} \n".format(sum(train_y == 0)))

print("After Under Sampling, counts of label 'Yes': {}".format(sum(train_y_under == 1)))
print("After Under Sampling, counts of label 'No': {} \n".format(sum(train_y_under == 0)))

print("After Under Sampling, the shape of train_X: {}".format(train_X_under.shape))
print("After Under Sampling, the shape of train_y: {} \n".format(train_y_under.shape))

### Logistic Regression on Undersampled Data

In [ ]:
undersampled_logit = LogisticRegression(random_state=1)
undersampled_logit.fit(train_X_under, train_y_under)

**Cross-validated recall check**

- Same K-fold approach, now run on the undersampled training data.


In [ ]:
cv_metric = "recall"
cv_splitter = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=1
)
under_cv_scores = cross_val_score(
    estimator=undersampled_logit, X=train_X_under, y=train_y_under, scoring=cv_metric, cv=cv_splitter
)
plt.boxplot(under_cv_scores)
plt.show()

* Training-fold recall ranges roughly between 0.70 and 0.76, similar to the oversampled model.
* Next, check validation performance.


In [ ]:
under_train_metrics = get_classification_metrics(
    undersampled_logit, train_X_under, train_y_under
)
print("Training performance:")
under_train_metrics

In [ ]:
under_val_metrics = get_classification_metrics(
    undersampled_logit, val_X, val_y
)
print("Validation performance:")
under_val_metrics

In [ ]:
plot_confusion_heatmap(undersampled_logit, val_X, val_y)

* This model generalizes well between train and validation.
* Undersampling improved performance overall - the model now separates the two classes noticeably better.


In [ ]:
# Compare training performance across all four model variants
train_comparison = pd.concat(
    [
        base_train_metrics.T,
        over_train_metrics.T,
        tuned_train_metrics.T,
        under_train_metrics.T,
    ],
    axis=1,
)
train_comparison.columns = [
    "Logistic Regression",
    "Logistic Regression with oversampled data",
    "Regularised Logistic Regression",
    "Logistic Regression with undersampled data",
]
print("Training performance comparison:")
train_comparison

In [ ]:
# Compare validation performance across all four model variants
val_comparison = pd.concat(
    [
        base_val_metrics.T,
        over_val_metrics.T,
        tuned_val_metrics.T,
        under_val_metrics.T,
    ],
    axis=1,
)
val_comparison.columns = [
    "Logistic Regression",
    "Logistic Regression with oversampled data",
    "Regularised Logistic Regression",
    "Logistic Regression with undersampled data",
]
print("Validation performance comparison:")
val_comparison

* The baseline logistic regression (no sampling, no regularization) generalizes the most consistently.

### Checking the Chosen Model Against the Test Set

In [ ]:
test_metrics = get_classification_metrics(
    undersampled_logit, test_X, test_y
)
print("Test performance:")
test_metrics

In [ ]:
plot_confusion_heatmap(base_logit, test_X, test_y)

- The model performs consistently on the held-out test set as well.

### Step 9: Interpreting the Coefficients

In [ ]:
# Coefficients and intercept of the fitted logistic regression
coef_table = pd.DataFrame(
    np.append(base_logit.coef_, base_logit.intercept_),
    index=train_X.columns.tolist() + ["Intercept"],
    columns=["Coefficients"],
)
coef_table.T

### Reading the Coefficients

* Positive coefficients (e.g. `gender_Male`, `purpose_House`, `purpose_Medical`) push the odds of delinquency up.
* Negative coefficients (e.g. `term_60 months`, `purpose_Other`, `home_ownership_Rent`) push the odds of delinquency down.


### Converting Coefficients to Odds

* Logistic regression coefficients are on the log-odds scale; exponentiating recovers the odds ratio.
* Therefore, **odds = exp(b)**
* And the percentage change in odds is **percent_change = (exp(b) - 1) * 100**


* Odds derived from the coefficients

In [ ]:
odds_table = np.exp(base_logit.coef_[0])  # convert log-odds coefficients to odds ratios
pd.set_option(
    "display.max_columns", None
)  # show every column
pd.DataFrame(
    odds_table, train_X.columns, columns=["odds"]
).T

* Percentage change in odds

In [ ]:
odds_pct_change = (np.exp(base_logit.coef_[0]) - 1) * 100  # percentage change in odds
pd.set_option(
    "display.max_columns", None
)
pd.DataFrame(
    odds_pct_change, train_X.columns, columns=["change_odds%"]
).T

### Conclusion

* **term**: Holding everything else constant, a 60-month term lowers the odds of delinquency by about
  87% relative to a shorter term.

* **gender**: A male borrower's odds of delinquency run about 2.21x (121% higher) those of a female borrower.

* **purpose**:
  * Borrowers seeking a house loan have about 1.09x (9.97% higher) odds of delinquency versus a car loan (car as reference).
  * Borrowers seeking a loan for medical reasons have about 1.52x (52% higher) odds of delinquency versus a car loan.
  * Borrowers seeking a loan for other reasons have roughly 3.9% lower odds of delinquency versus a car loan.

`The remaining categories can be interpreted the same way.`

* **age**: Holding everything else constant, being in the >25 age bracket is associated with roughly 75.9% higher
  odds of delinquency compared with the 20-25 bracket.

* **FICO**: A FICO score above 500 is associated with roughly 86% lower odds of delinquency compared with the
  300-500 bracket.


## Business Recommendations

* FICO bracket, loan term, and gender stand out as the most influential drivers of delinquency risk.
* Applicants requesting a 36-month term with a FICO score in the 300-500 range represent the highest risk and
  should not be approved under current criteria.
* Female applicants with a FICO score above 500 are the strongest candidate segment to target.
* A simple decision rule based on FICO score, loan duration, and gender captures most of the risk signal: if the
  FICO score is below 500 and the term is under 60 months, repayment risk is high; if the FICO score is above 500
  and the applicant is female, repayment odds are much better.
